# SensiFake Qwen3-VL throughput benchmark

Performance-only comparison of batch size 1 and 2 per GPU using two independent FP16 Qwen3-VL replicas on Kaggle Tesla T4 x2. The frozen rubric, prompt, model revision, JSON schema, and zero-shot behavior remain unchanged.

## 1. Imports

Pin the Transformers and Hub versions used by the completed pilot, then import the minimal PyTorch inference stack.

In [ ]:
%pip install -q "transformers==5.16.1" "huggingface_hub==1.29.0" accelerate

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import random
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import huggingface_hub
import pandas as pd
import torch
import transformers
from IPython.display import display
from PIL import Image, ImageOps
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, set_seed

## 2. Globals

The model revision is pinned and never resolved from main. Input paths use Kaggle's current mounted-dataset layout and point only to the canonical OpenFake pilot tree.

In [ ]:
SEED = 42
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
MODEL_REVISION = "ebb281ec70b05090aa6165b016eac8ec08e71b17"
MODEL_DTYPE = torch.float16
EXPECTED_GOLD_SHA256 = "c783a28509ca200d37d7b9e0f79a4bdd359c3f59f16acf2a0a80759875eb573f"
EXPECTED_GOLD_ROWS = 101
EXPECTED_MANIFEST_ROWS = 600
BENCHMARK_IMAGES_PER_LEVEL = 9
EXPECTED_BENCHMARK_IMAGES = 27
GPU_COUNT = 2
MAX_IMAGE_SIDE = 896

SOURCE_DATASET_ROOT = Path(
    "/kaggle/input/datasets/saracristinabasco/sensifake600"
)
PILOT_ROOT = Path(
    "/kaggle/input/datasets/saracristinabasco/"
    "sensifake600/datasets/datasets/openfake/pilot-600"
)
MANIFEST_PATH = PILOT_ROOT / "manifest.jsonl"
LEGACY_ROOT = SOURCE_DATASET_ROOT / "sensifake-600"
GOLD_CSV_PATH = Path(
    "/kaggle/input/datasets/saracristinabasco/"
    "sensifake-gold-development/sensitivity_annotations.csv"
)

OUTPUT_DIR = Path("/kaggle/working/qwen3vl_throughput_benchmark")
RESULTS_PATH = OUTPUT_DIR / "benchmark_results.csv"
SUMMARY_PATH = OUTPUT_DIR / "benchmark_summary.json"
CONFIG_A_PATH = OUTPUT_DIR / "config_a_predictions.csv"
CONFIG_B_PATH = OUTPUT_DIR / "config_b_predictions.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_TOTAL_INFERENCE_SECONDS = 3353.603
BASELINE_IMAGE_COUNT = 101
BASELINE_SECONDS_PER_IMAGE = (
    BASELINE_TOTAL_INFERENCE_SECONDS / BASELINE_IMAGE_COUNT
)
GENERATION_PARAMETERS = {
    "max_new_tokens": 220,
    "do_sample": False,
    "use_cache": True,
}
BENCHMARK_CONFIGS = (
    {"config_id": "A", "batch_size_per_gpu": 1, "predictions_path": CONFIG_A_PATH},
    {"config_id": "B", "batch_size_per_gpu": 2, "predictions_path": CONFIG_B_PATH},
)
LEVEL_ORDER = ("low", "medium", "high")

GOLD_REQUIRED_COLUMNS = (
    "content_hash", "blind_id", "public_relevance", "harm_urgency",
    "vulnerability", "sensitivity_score", "sensitivity_level",
    "sensitivity_rationale", "annotation_confidence", "needs_review",
    "annotation_round", "annotated_at",
)
PREDICTION_COLUMNS = (
    "content_hash", "sample_position", "config_id", "worker_id",
    "cuda_device", "batch_size_per_gpu", "batch_index", "batch_position",
    "batch_image_count", "pred_public_relevance", "pred_harm_urgency",
    "pred_vulnerability", "pred_sensitivity_score", "pred_sensitivity_level",
    "pred_sensitivity_rationale", "pred_annotation_confidence", "parse_ok",
    "parse_error", "raw_model_output", "batch_inference_seconds",
    "batch_seconds_per_image",
)

PROMPT_TEXT = """You are annotating the semantic sensitivity of visible image content.
Judge only what is visibly supported by the image. Do not determine whether the image is real or synthetic. Do not infer the dataset source, provenance, generator, model, prompt, caption, or any unseen context. Do not infer political, medical, conflict, or emergency context unless it is visually supported. Assess the potential sensitivity if imagery with the visible content were deceptive.

Use this frozen rubric:
public_relevance: 0 = private/everyday content; 1 = public event, profession, institution, or recognizable public context; 2 = politics, elections, armed conflict, disaster response, or public health.
harm_urgency: 0 = no plausible immediate public harm; 1 = reputational harm or general public confusion; 2 = physical safety, democratic process, emergency response, or health behavior could be affected.
vulnerability: 0 = no visibly vulnerable/distressed subjects; 1 = patients, displaced people, injured persons, children in crisis, or populations under threat.

Predict only the three component values. Do not calculate or output a total score or sensitivity level. Return exactly one compact JSON object, with no Markdown and no extra keys:
{"public_relevance": 0, "harm_urgency": 0, "vulnerability": 0, "sensitivity_rationale": "short rationale using only visible evidence", "annotation_confidence": "high"}
The component values must be integers in their stated ranges. The rationale must be concise (at most 280 characters). annotation_confidence must be exactly low, medium, or high."""

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
NOTEBOOK_STARTED_PERF = time.perf_counter()

## 3. Utils

Utilities preserve the pilot's strict parser, keep preprocessing in memory, execute one worker per CUDA device, and measure synchronized concurrent wall time and peak VRAM.

In [ ]:
@dataclass(frozen=True)
class BenchmarkItem:
    sample_position: int
    content_hash: str
    image: Image.Image


@dataclass
class ModelReplica:
    worker_id: int
    device: torch.device
    processor: Any
    model: Any


def utc_now() -> str:
    return datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def seeded_order_key(content_hash: str, purpose: str) -> str:
    value = f"{SEED}:{purpose}:{content_hash}".encode()
    return hashlib.sha256(value).hexdigest()


def derived_level(score: int) -> str:
    if score not in range(6):
        raise ValueError(f"sensitivity score outside 0..5: {score}")
    if score <= 1:
        return "low"
    if score <= 3:
        return "medium"
    return "high"


def reject_duplicate_json_keys(pairs: list[tuple[str, Any]]) -> dict[str, Any]:
    result: dict[str, Any] = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"duplicate JSON key: {key}")
        result[key] = value
    return result


def parse_model_json(raw_output: str) -> dict[str, Any]:
    expected = {
        "public_relevance", "harm_urgency", "vulnerability",
        "sensitivity_rationale", "annotation_confidence",
    }
    payload = json.loads(
        raw_output.strip(), object_pairs_hook=reject_duplicate_json_keys
    )
    if not isinstance(payload, dict):
        raise TypeError("response must be one JSON object")
    if set(payload) != expected:
        missing = sorted(expected - set(payload))
        extra = sorted(set(payload) - expected)
        raise ValueError(
            f"JSON keys do not match schema; missing={missing}, extra={extra}"
        )

    ranges = {
        "public_relevance": range(3),
        "harm_urgency": range(3),
        "vulnerability": range(2),
    }
    for field, allowed in ranges.items():
        value = payload[field]
        if type(value) is not int or value not in allowed:
            raise ValueError(f"{field} must be an integer in {list(allowed)}")

    rationale = payload["sensitivity_rationale"]
    if not isinstance(rationale, str) or not rationale.strip():
        raise ValueError("sensitivity_rationale must be a non-empty string")
    if len(rationale.strip()) > 280:
        raise ValueError("sensitivity_rationale exceeds 280 characters")
    confidence = payload["annotation_confidence"]
    if confidence not in {"low", "medium", "high"}:
        raise ValueError("annotation_confidence must be low, medium, or high")

    payload["sensitivity_rationale"] = rationale.strip()
    score = (
        payload["public_relevance"]
        + payload["harm_urgency"]
        + payload["vulnerability"]
    )
    payload["sensitivity_score"] = score
    payload["sensitivity_level"] = derived_level(score)
    return payload


def load_image_in_memory(path: Path) -> Image.Image:
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")
        if max(image.size) > MAX_IMAGE_SIDE:
            image.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE), Image.Resampling.LANCZOS)
        return image.copy()


def atomic_write_csv(frame: pd.DataFrame, path: Path) -> None:
    temporary = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_json(payload: dict[str, Any], path: Path) -> None:
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    os.replace(temporary, path)


def synchronize_devices() -> None:
    for device_index in range(GPU_COUNT):
        torch.cuda.synchronize(device_index)


def gib(byte_count: int) -> float:
    return byte_count / (1024**3)


def blind_messages(image: Image.Image) -> list[dict[str, Any]]:
    # The model receives only pixels and the unchanged prompt.
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": PROMPT_TEXT},
            ],
        }
    ]


def item_batches(
    items: list[BenchmarkItem], batch_size: int
) -> list[list[BenchmarkItem]]:
    return [items[start : start + batch_size] for start in range(0, len(items), batch_size)]


def infer_batch(
    replica: ModelReplica,
    items: list[BenchmarkItem],
    config_id: str,
    batch_size_per_gpu: int,
    batch_index: int,
) -> list[dict[str, Any]]:
    torch.cuda.set_device(replica.device)
    conversations = [blind_messages(item.image) for item in items]
    torch.cuda.synchronize(replica.device)
    batch_started = time.perf_counter()
    inputs = replica.processor.apply_chat_template(
        conversations,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        padding=True,
    )
    inputs.pop("token_type_ids", None)
    inputs = inputs.to(replica.device)
    input_length = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        generated = replica.model.generate(**inputs, **GENERATION_PARAMETERS)
    torch.cuda.synchronize(replica.device)
    batch_seconds = time.perf_counter() - batch_started
    generated_only = generated[:, input_length:]
    raw_outputs = replica.processor.batch_decode(
        generated_only,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    records: list[dict[str, Any]] = []
    for batch_position, (item, raw_output) in enumerate(zip(items, raw_outputs, strict=True)):
        raw_output = raw_output.strip()
        try:
            parsed = parse_model_json(raw_output)
            parse_ok = True
            parse_error = ""
        except (json.JSONDecodeError, TypeError, ValueError) as exc:
            parsed = None
            parse_ok = False
            parse_error = f"{type(exc).__name__}: {exc}"
        records.append(
            {
                "content_hash": item.content_hash,
                "sample_position": item.sample_position,
                "config_id": config_id,
                "worker_id": replica.worker_id,
                "cuda_device": str(replica.device),
                "batch_size_per_gpu": batch_size_per_gpu,
                "batch_index": batch_index,
                "batch_position": batch_position,
                "batch_image_count": len(items),
                "pred_public_relevance": parsed["public_relevance"] if parsed else None,
                "pred_harm_urgency": parsed["harm_urgency"] if parsed else None,
                "pred_vulnerability": parsed["vulnerability"] if parsed else None,
                "pred_sensitivity_score": parsed["sensitivity_score"] if parsed else None,
                "pred_sensitivity_level": parsed["sensitivity_level"] if parsed else None,
                "pred_sensitivity_rationale": (
                    parsed["sensitivity_rationale"] if parsed else None
                ),
                "pred_annotation_confidence": (
                    parsed["annotation_confidence"] if parsed else None
                ),
                "parse_ok": parse_ok,
                "parse_error": parse_error,
                "raw_model_output": raw_output,
                "batch_inference_seconds": round(batch_seconds, 6),
                "batch_seconds_per_image": round(batch_seconds / len(items), 6),
            }
        )
    del inputs, generated, generated_only, conversations, raw_outputs
    return records


def worker_run(
    replica: ModelReplica,
    assigned_items: list[BenchmarkItem],
    config_id: str,
    batch_size_per_gpu: int,
    ready_barrier: threading.Barrier,
    start_event: threading.Event,
) -> list[dict[str, Any]]:
    torch.cuda.set_device(replica.device)
    torch.cuda.synchronize(replica.device)
    ready_barrier.wait()
    start_event.wait()
    records: list[dict[str, Any]] = []
    for batch_index, batch in enumerate(item_batches(assigned_items, batch_size_per_gpu)):
        records.extend(
            infer_batch(replica, batch, config_id, batch_size_per_gpu, batch_index)
        )
    return records


def warm_up_replicas(
    replicas: list[ModelReplica],
    assignments: list[list[BenchmarkItem]],
    batch_size_per_gpu: int,
) -> None:
    with ThreadPoolExecutor(max_workers=GPU_COUNT) as executor:
        futures = [
            executor.submit(
                infer_batch,
                replica,
                assignments[replica.worker_id][:batch_size_per_gpu],
                "warmup",
                batch_size_per_gpu,
                -1,
            )
            for replica in replicas
        ]
        for future in futures:
            future.result()
    synchronize_devices()


def run_benchmark_configuration(
    replicas: list[ModelReplica],
    items: tuple[BenchmarkItem, ...],
    config_id: str,
    batch_size_per_gpu: int,
    predictions_path: Path,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    assignments = [list(items[worker_id::GPU_COUNT]) for worker_id in range(GPU_COUNT)]
    assigned_hashes = [item.content_hash for group in assignments for item in group]
    expected_hashes = [item.content_hash for item in items]
    if len(assigned_hashes) != len(set(assigned_hashes)) or set(assigned_hashes) != set(expected_hashes):
        raise RuntimeError("dual-GPU work split is incomplete or duplicated")

    # Configuration-boundary cache cleanup keeps reserved-VRAM peaks comparable.
    synchronize_devices()
    for device_index in range(GPU_COUNT):
        with torch.cuda.device(device_index):
            torch.cuda.empty_cache()

    # One matching-shape warm-up batch per replica; excluded from benchmark timing.
    warm_up_replicas(replicas, assignments, batch_size_per_gpu)
    for device_index in range(GPU_COUNT):
        torch.cuda.reset_peak_memory_stats(device_index)
    synchronize_devices()

    ready_barrier = threading.Barrier(GPU_COUNT + 1)
    start_event = threading.Event()
    with ThreadPoolExecutor(max_workers=GPU_COUNT) as executor:
        futures = [
            executor.submit(
                worker_run,
                replica,
                assignments[replica.worker_id],
                config_id,
                batch_size_per_gpu,
                ready_barrier,
                start_event,
            )
            for replica in replicas
        ]
        ready_barrier.wait()
        synchronize_devices()
        wall_started = time.perf_counter()
        start_event.set()
        worker_outputs = [future.result() for future in futures]
        synchronize_devices()
        wall_seconds = time.perf_counter() - wall_started

    records = [record for worker_records in worker_outputs for record in worker_records]
    frame = pd.DataFrame(records, columns=PREDICTION_COLUMNS).sort_values(
        "sample_position"
    ).reset_index(drop=True)
    if len(frame) != EXPECTED_BENCHMARK_IMAGES:
        raise RuntimeError(f"configuration {config_id} produced {len(frame)} rows, expected 27")
    if frame["content_hash"].duplicated().any() or set(frame["content_hash"]) != set(expected_hashes):
        raise RuntimeError(f"configuration {config_id} output hashes are incomplete or duplicated")
    atomic_write_csv(frame, predictions_path)

    parse_success_count = int(frame["parse_ok"].sum())
    summary = {
        "config_id": config_id,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "dtype": "float16",
        "max_image_side": MAX_IMAGE_SIDE,
        "batch_size_per_gpu": batch_size_per_gpu,
        "gpu_count": GPU_COUNT,
        "images_processed": len(frame),
        "total_wall_clock_seconds": wall_seconds,
        "images_per_second": len(frame) / wall_seconds,
        "seconds_per_image": wall_seconds / len(frame),
        "baseline_seconds_per_image": BASELINE_SECONDS_PER_IMAGE,
        "speedup_vs_observed_baseline": BASELINE_SECONDS_PER_IMAGE / (wall_seconds / len(frame)),
        "parse_success_count": parse_success_count,
        "parse_failure_count": len(frame) - parse_success_count,
        "parse_success_rate": parse_success_count / len(frame),
        "peak_allocated_gib_gpu_0": gib(torch.cuda.max_memory_allocated(0)),
        "peak_reserved_gib_gpu_0": gib(torch.cuda.max_memory_reserved(0)),
        "peak_allocated_gib_gpu_1": gib(torch.cuda.max_memory_allocated(1)),
        "peak_reserved_gib_gpu_1": gib(torch.cuda.max_memory_reserved(1)),
        "warmup_batches_per_gpu": 1,
        "warmup_excluded_from_timing": True,
        "predictions_path": str(predictions_path),
    }
    return frame, summary

## 4. Data

Select 9 Gold LOW, 9 MEDIUM, and all 9 HIGH cases deterministically with seed 42. Human labels are used only for sampling. The resulting inference objects contain only an opaque host-side hash and in-memory pixels.

In [ ]:
if not GOLD_CSV_PATH.is_file():
    raise FileNotFoundError(f"Gold annotations not found: {GOLD_CSV_PATH}")
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"canonical pilot manifest not found: {MANIFEST_PATH}")
if sha256_file(GOLD_CSV_PATH) != EXPECTED_GOLD_SHA256:
    raise RuntimeError("Gold CSV SHA-256 does not match the frozen artifact")

gold_frame = pd.read_csv(GOLD_CSV_PATH, keep_default_na=False)
if tuple(gold_frame.columns) != GOLD_REQUIRED_COLUMNS:
    raise RuntimeError("Gold CSV columns do not exactly match the frozen schema")
if len(gold_frame) != EXPECTED_GOLD_ROWS:
    raise RuntimeError(f"expected 101 Gold rows, found {len(gold_frame)}")
if gold_frame["content_hash"].duplicated().any():
    raise RuntimeError("Gold content_hash values are not unique")

gold_distribution = gold_frame["sensitivity_level"].value_counts().to_dict()
if gold_distribution.get("high") != BENCHMARK_IMAGES_PER_LEVEL:
    raise RuntimeError(
        f"expected exactly 9 Gold HIGH cases, found {gold_distribution.get('high', 0)}"
    )
selected_rows: list[dict[str, str]] = []
for level in LEVEL_ORDER:
    candidates = gold_frame.loc[
        gold_frame["sensitivity_level"] == level, ["content_hash", "sensitivity_level"]
    ].to_dict("records")
    if len(candidates) < BENCHMARK_IMAGES_PER_LEVEL:
        raise RuntimeError(f"not enough Gold {level.upper()} cases for the benchmark")
    ordered = sorted(
        candidates,
        key=lambda row: seeded_order_key(str(row["content_hash"]), f"select-{level}"),
    )
    # Exactly nine HIGH cases means all of them are retained.
    selected_rows.extend(ordered[:BENCHMARK_IMAGES_PER_LEVEL])

selected_rows = sorted(
    selected_rows,
    key=lambda row: seeded_order_key(str(row["content_hash"]), "benchmark-order"),
)
selection_frame = pd.DataFrame(selected_rows)
BENCHMARK_SAMPLE_DISTRIBUTION = (
    selection_frame["sensitivity_level"]
    .value_counts()
    .reindex(LEVEL_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)
if BENCHMARK_SAMPLE_DISTRIBUTION != {"low": 9, "medium": 9, "high": 9}:
    raise RuntimeError(f"benchmark sample is not 9/9/9: {BENCHMARK_SAMPLE_DISTRIBUTION}")

manifest_records: list[dict[str, Any]] = []
with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            raise RuntimeError(f"blank manifest line at {line_number}")
        try:
            manifest_records.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise RuntimeError(f"invalid manifest JSON at line {line_number}") from exc
if len(manifest_records) != EXPECTED_MANIFEST_ROWS:
    raise RuntimeError(f"expected 600 manifest rows, found {len(manifest_records)}")
manifest_frame = pd.DataFrame(manifest_records)
if not {"content_hash", "relative_image_path"}.issubset(manifest_frame.columns):
    raise RuntimeError("pilot manifest lacks path-resolution fields")

selected_hashes = set(selection_frame["content_hash"].astype(str))
manifest_matches = manifest_frame[manifest_frame["content_hash"].isin(selected_hashes)][
    ["content_hash", "relative_image_path"]
].copy()
match_counts = manifest_matches["content_hash"].value_counts()
bad_counts = {
    content_hash: int(match_counts.get(content_hash, 0))
    for content_hash in selected_hashes
    if match_counts.get(content_hash, 0) != 1
}
if bad_counts:
    raise RuntimeError(f"selected hashes do not resolve exactly once: {bad_counts}")

path_resolution = selection_frame[["content_hash"]].merge(
    manifest_matches, on="content_hash", how="left", validate="one_to_one"
)
pilot_root_resolved = PILOT_ROOT.resolve()
legacy_root_resolved = LEGACY_ROOT.resolve(strict=False)
benchmark_items_list: list[BenchmarkItem] = []
resolved_paths: set[Path] = set()
for sample_position, row in enumerate(path_resolution.itertuples(index=False)):
    image_path = (PILOT_ROOT / str(row.relative_image_path)).resolve()
    if not image_path.is_relative_to(pilot_root_resolved):
        raise RuntimeError(f"resolved path escapes canonical pilot root: {image_path}")
    if image_path.is_relative_to(legacy_root_resolved):
        raise RuntimeError(f"legacy duplicate path is forbidden: {image_path}")
    if not image_path.is_file():
        raise FileNotFoundError(f"benchmark image is missing: {image_path}")
    if image_path in resolved_paths:
        raise RuntimeError(f"duplicate resolved image path: {image_path}")
    resolved_paths.add(image_path)
    benchmark_items_list.append(
        BenchmarkItem(
            sample_position=sample_position,
            content_hash=str(row.content_hash),
            image=load_image_in_memory(image_path),
        )
    )

benchmark_items = tuple(benchmark_items_list)
if len(benchmark_items) != EXPECTED_BENCHMARK_IMAGES:
    raise RuntimeError(f"resolved {len(benchmark_items)} images, expected 27")

# Enforce the sampling/inference boundary: no per-image human label or path survives.
del (
    gold_frame, selected_rows, selection_frame, manifest_records, manifest_frame,
    manifest_matches, path_resolution, benchmark_items_list, resolved_paths,
)
print(f"Validated and preloaded {len(benchmark_items)} images at max side {MAX_IMAGE_SIDE}.")
print(f"Sampling strata: {BENCHMARK_SAMPLE_DISTRIBUTION}")

## 5. Network

Load two complete, independent unquantized FP16 replicas: one wholly on cuda:0 and one wholly on cuda:1. No automatic sharding or CPU/disk offload is permitted.

In [ ]:
if transformers.__version__ != "5.16.1":
    raise RuntimeError(f"unexpected Transformers version: {transformers.__version__}")
if huggingface_hub.__version__ != "1.29.0":
    raise RuntimeError(f"unexpected huggingface_hub version: {huggingface_hub.__version__}")
if not torch.cuda.is_available() or torch.cuda.device_count() < GPU_COUNT:
    raise RuntimeError("This benchmark requires two CUDA GPUs")
gpu_names = [torch.cuda.get_device_name(index) for index in range(GPU_COUNT)]
if any("T4" not in name for name in gpu_names):
    raise RuntimeError(f"This benchmark is restricted to Tesla T4 x2; found {gpu_names}")


def normalized_device_name(value: Any) -> str:
    if isinstance(value, int):
        return f"cuda:{value}"
    return str(value)


def verify_complete_replica(replica: ModelReplica) -> None:
    intended = str(replica.device)
    parameter_devices = {str(parameter.device) for parameter in replica.model.parameters()}
    buffer_devices = {str(buffer.device) for buffer in replica.model.buffers()}
    if parameter_devices != {intended}:
        raise RuntimeError(
            f"worker {replica.worker_id} parameters are not wholly on {intended}: "
            f"{sorted(parameter_devices)}"
        )
    if not buffer_devices.issubset({intended}):
        raise RuntimeError(
            f"worker {replica.worker_id} buffers escaped {intended}: {sorted(buffer_devices)}"
        )
    floating_dtypes = {
        parameter.dtype
        for parameter in replica.model.parameters()
        if parameter.is_floating_point()
    }
    if floating_dtypes != {MODEL_DTYPE}:
        raise RuntimeError(
            f"worker {replica.worker_id} parameter dtypes are not pure FP16: {floating_dtypes}"
        )
    if getattr(replica.model, "is_quantized", False):
        raise RuntimeError("quantization is forbidden for this benchmark")
    hf_device_map = getattr(replica.model, "hf_device_map", {})
    mapped_devices = {normalized_device_name(value) for value in hf_device_map.values()}
    if mapped_devices and mapped_devices != {intended}:
        raise RuntimeError(
            f"worker {replica.worker_id} has sharding or offload targets: {mapped_devices}"
        )
    if mapped_devices.intersection({"cpu", "disk"}):
        raise RuntimeError("CPU or disk offload is forbidden")


def load_replica(worker_id: int) -> ModelReplica:
    device = torch.device(f"cuda:{worker_id}")
    torch.cuda.set_device(device)
    processor = AutoProcessor.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False
    )
    processor.tokenizer.padding_side = "left"
    if processor.tokenizer.pad_token_id is None:
        raise RuntimeError("Qwen tokenizer has no pad token for batched generation")
    try:
        model = Qwen3VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            revision=MODEL_REVISION,
            dtype=MODEL_DTYPE,
            device_map={"": str(device)},
            attn_implementation="sdpa",
            low_cpu_mem_usage=True,
            trust_remote_code=False,
        )
    except torch.OutOfMemoryError as exc:
        torch.cuda.empty_cache()
        raise RuntimeError(
            f"One unquantized FP16 replica does not fit on {device}; "
            "stop without sharding or quantization."
        ) from exc
    model.eval()
    replica = ModelReplica(worker_id, device, processor, model)
    verify_complete_replica(replica)
    return replica


replicas = [load_replica(worker_id) for worker_id in range(GPU_COUNT)]
if replicas[0].model is replicas[1].model:
    raise RuntimeError("GPU workers unexpectedly share one model object")
synchronize_devices()
loaded_vram_gib = {
    f"gpu_{index}": gib(torch.cuda.memory_allocated(index))
    for index in range(GPU_COUNT)
}
print(f"Loaded independent FP16 replicas on {gpu_names}; allocated VRAM={loaded_vram_gib}")

## 6. Train / Inference Benchmark

There is no training and model weights are never updated. Two Python worker threads overlap inference on separate model replicas and CUDA devices. Warm-up is excluded; synchronized wall time includes prompt/image processing and generation for all 27 preloaded images.

In [ ]:
configuration_frames: dict[str, pd.DataFrame] = {}
configuration_summaries: list[dict[str, Any]] = []

for configuration in BENCHMARK_CONFIGS:
    config_id = str(configuration["config_id"])
    batch_size_per_gpu = int(configuration["batch_size_per_gpu"])
    predictions_path = Path(configuration["predictions_path"])
    print(
        f"Running CONFIG {config_id}: two concurrent GPUs, "
        f"batch size {batch_size_per_gpu} per GPU"
    )
    try:
        frame, summary = run_benchmark_configuration(
            replicas=replicas,
            items=benchmark_items,
            config_id=config_id,
            batch_size_per_gpu=batch_size_per_gpu,
            predictions_path=predictions_path,
        )
    except torch.OutOfMemoryError as exc:
        synchronize_devices()
        for device_index in range(GPU_COUNT):
            with torch.cuda.device(device_index):
                torch.cuda.empty_cache()
        raise RuntimeError(
            f"CONFIG {config_id} ran out of GPU memory; no fallback, sharding, "
            "or quantization was attempted."
        ) from exc
    configuration_frames[config_id] = frame
    configuration_summaries.append(summary)
    atomic_write_csv(pd.DataFrame(configuration_summaries), RESULTS_PATH)
    print(
        f"CONFIG {config_id}: {summary['seconds_per_image']:.3f} s/image, "
        f"{summary['images_per_second']:.3f} images/s, "
        f"speedup={summary['speedup_vs_observed_baseline']:.2f}x, "
        f"valid JSON={summary['parse_success_count']}/{summary['images_processed']}"
    )

benchmark_results = pd.DataFrame(configuration_summaries)
if set(configuration_frames) != {"A", "B"}:
    raise RuntimeError("both benchmark configurations must complete before evaluation")
display(benchmark_results)

## 7. Evaluation / Benchmark Summary

Summarize throughput, memory, parse validity, and cross-batch determinism only. No annotation-quality conclusion or pass/fail threshold is introduced.

In [ ]:
config_a = configuration_frames["A"].copy()
config_b = configuration_frames["B"].copy()
comparison_columns = [
    "content_hash", "parse_ok", "pred_public_relevance", "pred_harm_urgency",
    "pred_vulnerability", "pred_sensitivity_level",
]
comparison = config_a[comparison_columns].merge(
    config_b[comparison_columns],
    on="content_hash",
    how="inner",
    validate="one_to_one",
    suffixes=("_a", "_b"),
)
if len(comparison) != EXPECTED_BENCHMARK_IMAGES:
    raise RuntimeError("CONFIG A/B comparison did not contain the same 27 hashes")

valid_in_both = comparison["parse_ok_a"] & comparison["parse_ok_b"]
same_triplet = (
    (comparison["pred_public_relevance_a"] == comparison["pred_public_relevance_b"])
    & (comparison["pred_harm_urgency_a"] == comparison["pred_harm_urgency_b"])
    & (comparison["pred_vulnerability_a"] == comparison["pred_vulnerability_b"])
)
same_level = (
    comparison["pred_sensitivity_level_a"]
    == comparison["pred_sensitivity_level_b"]
)
sanity_diagnostic = {
    "images_compared": len(comparison),
    "valid_json_in_both": int(valid_in_both.sum()),
    "any_parse_failure": int((~valid_in_both).sum()),
    "identical_component_triplets": int((valid_in_both & same_triplet).sum()),
    "different_component_triplets": int((valid_in_both & ~same_triplet).sum()),
    "identical_sensitivity_levels": int((valid_in_both & same_level).sum()),
}
all_outputs_valid = all(
    summary["parse_success_count"] == EXPECTED_BENCHMARK_IMAGES
    for summary in configuration_summaries
)

benchmark_summary = {
    "benchmark_completed_at_utc": utc_now(),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_replicas": 2,
    "replica_devices": [str(replica.device) for replica in replicas],
    "execution": "two concurrent worker threads with one complete model replica per GPU",
    "gpu_names": gpu_names,
    "dtype": "float16",
    "quantization": None,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "huggingface_hub_version": huggingface_hub.__version__,
    "cuda_version": torch.version.cuda,
    "seed": SEED,
    "prompt_text": PROMPT_TEXT,
    "prompt_sha256": hashlib.sha256(PROMPT_TEXT.encode("utf-8")).hexdigest(),
    "generation_parameters": GENERATION_PARAMETERS,
    "image_preprocessing": {
        "maximum_image_side_pixels": MAX_IMAGE_SIDE,
        "preserve_aspect_ratio": True,
        "resize_location": "memory_only_before_timing",
        "resampling": "PIL.Image.Resampling.LANCZOS",
        "exif_transpose": True,
        "color_mode": "RGB",
        "processor": "AutoProcessor defaults from the pinned model revision",
    },
    "benchmark_sample": {
        "images": EXPECTED_BENCHMARK_IMAGES,
        "human_gold_distribution_used_for_selection_only": BENCHMARK_SAMPLE_DISTRIBUTION,
        "selection_seed": SEED,
        "selection_method": "SHA-256 ordering of seed, stratum, and content_hash",
    },
    "baseline": {
        "total_inference_seconds": BASELINE_TOTAL_INFERENCE_SECONDS,
        "images": BASELINE_IMAGE_COUNT,
        "seconds_per_image": BASELINE_SECONDS_PER_IMAGE,
    },
    "configurations": configuration_summaries,
    "sanity_diagnostic": sanity_diagnostic,
    "all_outputs_valid_json": all_outputs_valid,
    "notebook_runtime_seconds": time.perf_counter() - NOTEBOOK_STARTED_PERF,
    "artifacts": {
        "benchmark_results": str(RESULTS_PATH),
        "config_a_predictions": str(CONFIG_A_PATH),
        "config_b_predictions": str(CONFIG_B_PATH),
    },
}
atomic_write_json(benchmark_summary, SUMMARY_PATH)

display(
    benchmark_results[
        [
            "config_id", "batch_size_per_gpu", "images_processed",
            "total_wall_clock_seconds", "images_per_second", "seconds_per_image",
            "speedup_vs_observed_baseline", "parse_success_count",
            "peak_allocated_gib_gpu_0", "peak_allocated_gib_gpu_1",
            "peak_reserved_gib_gpu_0", "peak_reserved_gib_gpu_1",
        ]
    ]
)
display(pd.Series(sanity_diagnostic, name="value").to_frame())
if not all_outputs_valid:
    print("WARNING: at least one strict JSON parse failed; raw outputs were retained.")
    for config_id, frame in configuration_frames.items():
        failures = frame.loc[
            ~frame["parse_ok"], ["content_hash", "parse_error", "raw_model_output"]
        ]
        if not failures.empty:
            print(f"CONFIG {config_id} parse failures")
            display(failures)
print(f"Benchmark artifacts saved under {OUTPUT_DIR}")